# Step 4 - Add Weather And Compare Models

This notebook joins origin-airport weather to flights using a timezone-aware UTC join, then compares Random Forest flight-only versus flight plus weather.


In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

## Load Flight Data


In [16]:
data_dir = Path("../data/raw/flight/2022")
csv_files = sorted(data_dir.glob("2022-0[1-3]_flight.csv"))

flight_cols = [
    "Month",
    "FlightDate",
    "DayOfWeek",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepDelay",
    "Cancelled"
]

flight_parts = []

for file in csv_files:
    flight_parts.append(pd.read_csv(file, usecols=flight_cols))

flights = pd.concat(flight_parts, ignore_index=True)
flights.shape

(1598468, 9)

In [17]:
flights = flights[flights["Cancelled"] == 0].copy()

flights = flights.dropna(subset=[
    "FlightDate",
    "CRSDepTime",
    "DepDelay",
    "Reporting_Airline",
    "Origin",
    "Dest"
])

flights = flights.sample(n=200_000, random_state=42)

flights["Delay"] = (flights["DepDelay"] > 15).astype(int)
flights = flights.drop(columns=["DepDelay"])

flights["FlightDate"] = pd.to_datetime(flights["FlightDate"])
flights["CRSDepTime"] = flights["CRSDepTime"].astype(int)
flights["dep_hour"] = flights["CRSDepTime"] // 100

dep_minutes = (flights["CRSDepTime"] // 100) * 60 + (flights["CRSDepTime"] % 100)
flights["scheduled_departure_local"] = flights["FlightDate"] + pd.to_timedelta(dep_minutes, unit="m")

flights["route"] = flights["Origin"] + "_" + flights["Dest"]
flights["is_weekend"] = flights["DayOfWeek"].isin([6, 7]).astype(int)
flights["time_of_day_bin"] = pd.cut(
    flights["dep_hour"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["overnight", "morning", "afternoon", "evening", "night"]
)

flights.shape

(200000, 14)

## Load Weather Data


In [18]:
weather_path = Path("../data/raw/weather/asos_top20_origins_2022_01_to_2022_03.csv")
weather = pd.read_csv(weather_path)
weather["valid"] = pd.to_datetime(weather["valid"])
weather.shape

(51923, 15)

In [19]:
weather_stations = sorted(weather["station"].dropna().unique().tolist())
flights = flights[flights["Origin"].isin(weather_stations)].copy()

print("Flights after top-20 weather airport filter:", flights.shape)
print("Weather stations:", weather_stations)

Flights after top-20 weather airport filter: (104229, 14)
Weather stations: ['ATL', 'BOS', 'CLT', 'DCA', 'DEN', 'DFW', 'DTW', 'EWR', 'IAH', 'JFK', 'LAS', 'LAX', 'LGA', 'MCO', 'MIA', 'MSP', 'ORD', 'PHX', 'SEA', 'SFO']


## Convert Local Flight Time To UTC


In [20]:
AIRPORT_TIMEZONES = {
    "ATL": "America/New_York",
    "BOS": "America/New_York",
    "CLT": "America/New_York",
    "DCA": "America/New_York",
    "DTW": "America/New_York",
    "EWR": "America/New_York",
    "JFK": "America/New_York",
    "LGA": "America/New_York",
    "MCO": "America/New_York",
    "MIA": "America/New_York",
    "DFW": "America/Chicago",
    "IAH": "America/Chicago",
    "MSP": "America/Chicago",
    "ORD": "America/Chicago",
    "DEN": "America/Denver",
    "PHX": "America/Phoenix",
    "LAS": "America/Los_Angeles",
    "LAX": "America/Los_Angeles",
    "SEA": "America/Los_Angeles",
    "SFO": "America/Los_Angeles"
}

missing_tz = sorted(set(flights["Origin"].unique()) - set(AIRPORT_TIMEZONES.keys()))
missing_tz

[]

In [21]:
def add_scheduled_departure_utc(flight_df, timezone_map):
    pieces = []

    for airport, group in flight_df.groupby("Origin", group_keys=False):
        tz_name = timezone_map.get(airport)
        if tz_name is None:
            raise ValueError(f"Missing timezone for airport: {airport}")

        group = group.copy()
        localized = group["scheduled_departure_local"].dt.tz_localize(
            tz_name,
            nonexistent="shift_forward",
            ambiguous="NaT"
        )
        group["scheduled_departure_utc"] = localized.dt.tz_convert("UTC").dt.tz_localize(None)
        pieces.append(group)

    return pd.concat(pieces, ignore_index=True)


flights = add_scheduled_departure_utc(flights, AIRPORT_TIMEZONES)
flights[["Origin", "FlightDate", "scheduled_departure_local", "scheduled_departure_utc"]].head()

,Origin,FlightDate,scheduled_departure_local,scheduled_departure_utc
0,ATL,2022-01-19,2022-01-19 08:20:00,2022-01-19 13:20:00
1,ATL,2022-02-02,2022-02-02 18:07:00,2022-02-02 23:07:00
2,ATL,2022-01-21,2022-01-21 05:30:00,2022-01-21 10:30:00
3,ATL,2022-03-27,2022-03-27 16:43:00,2022-03-27 20:43:00
4,ATL,2022-01-11,2022-01-11 21:30:00,2022-01-12 02:30:00


## Join Weather


In [22]:
def join_weather_to_flights(flight_df, weather_df, tolerance_hours=2):
    joined_parts = []
    tolerance = pd.Timedelta(hours=tolerance_hours)

    for station in sorted(set(flight_df["Origin"]).intersection(set(weather_df["station"]))):
        flight_part = flight_df[flight_df["Origin"] == station].sort_values("scheduled_departure_utc").copy()
        weather_part = weather_df[weather_df["station"] == station].sort_values("valid").copy()

        merged = pd.merge_asof(
            flight_part,
            weather_part,
            left_on="scheduled_departure_utc",
            right_on="valid",
            direction="backward",
            tolerance=tolerance
        )
        joined_parts.append(merged)

    joined = pd.concat(joined_parts, ignore_index=True)
    joined["weather_report_age_minutes"] = (
        joined["scheduled_departure_utc"] - joined["valid"]
    ).dt.total_seconds().div(60)

    return joined


flights_weather = join_weather_to_flights(flights, weather, tolerance_hours=2)
flights_weather.shape

(104229, 31)

In [23]:
coverage = flights_weather["valid"].notna().mean()
print("Weather join coverage:", coverage)

flights_weather[[
    "Origin",
    "scheduled_departure_local",
    "scheduled_departure_utc",
    "valid",
    "weather_report_age_minutes",
    "tmpf",
    "sknt",
    "vsby",
    "wxcodes"
]].head()

Weather join coverage: 0.9985224841454874


,Origin,scheduled_departure_local,scheduled_departure_utc,valid,weather_report_age_minutes,tmpf,sknt,vsby,wxcodes
0,ATL,2022-01-01 06:30:00,2022-01-01 11:30:00,2022-01-01 10:52:00,38.0,70.0,7.0,10.0,NaN
1,ATL,2022-01-01 06:45:00,2022-01-01 11:45:00,2022-01-01 10:52:00,53.0,70.0,7.0,10.0,NaN
2,ATL,2022-01-01 08:00:00,2022-01-01 13:00:00,2022-01-01 12:52:00,8.0,70.0,9.0,10.0,NaN
3,ATL,2022-01-01 08:08:00,2022-01-01 13:08:00,2022-01-01 13:04:00,4.0,70.0,10.0,10.0,NaN
4,ATL,2022-01-01 08:25:00,2022-01-01 13:25:00,2022-01-01 13:04:00,21.0,70.0,10.0,10.0,NaN


## Build Weather Features


In [24]:
model_df = flights_weather.copy()

model_df["gust"] = model_df["gust"].fillna(0)
model_df["p01i"] = model_df["p01i"].fillna(0)
model_df["vsby"] = model_df["vsby"].fillna(10)
model_df["wxcodes"] = model_df["wxcodes"].fillna("")
model_df["skyc1"] = model_df["skyc1"].fillna("CLR")

model_df["has_precip"] = (model_df["p01i"] > 0).astype(int)
model_df["low_visibility"] = (model_df["vsby"] < 3).astype(int)
model_df["high_wind"] = (model_df["sknt"].fillna(0) >= 15).astype(int)
model_df["has_gust"] = (model_df["gust"] > 0).astype(int)
model_df["has_weather_code"] = (model_df["wxcodes"] != "").astype(int)

required_weather_cols = ["valid", "tmpf", "dwpf", "relh", "sknt", "alti", "weather_report_age_minutes"]
model_df = model_df.dropna(subset=required_weather_cols).copy()

model_df.shape

(103974, 36)

## Time-Based Train/Test Split


In [25]:
train_df = model_df[model_df["Month"].isin([1, 2])].copy()
test_df = model_df[model_df["Month"] == 3].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train delay rate:", train_df["Delay"].mean())
print("Test delay rate:", test_df["Delay"].mean())

Train shape: (66459, 36)
Test shape: (37515, 36)
Train delay rate: 0.19925066582404188
Test delay rate: 0.21052912168465948


## Historical Delay Rate Features


In [26]:
def add_delay_rate_feature(train_data, test_data, group_col, target_col, new_col):
    global_rate = train_data[target_col].mean()
    rates = train_data.groupby(group_col)[target_col].mean()

    train_data[new_col] = train_data[group_col].map(rates).fillna(global_rate)
    test_data[new_col] = test_data[group_col].map(rates).fillna(global_rate)

    return train_data, test_data


train_df, test_df = add_delay_rate_feature(train_df, test_df, "Origin", "Delay", "origin_delay_rate")
train_df, test_df = add_delay_rate_feature(train_df, test_df, "Reporting_Airline", "Delay", "airline_delay_rate")
train_df, test_df = add_delay_rate_feature(train_df, test_df, "route", "Delay", "route_delay_rate")

train_df[["Origin", "Reporting_Airline", "route", "origin_delay_rate", "airline_delay_rate", "route_delay_rate"]].head()

,Origin,Reporting_Airline,route,origin_delay_rate,airline_delay_rate,route_delay_rate
0,ATL,B6,ATL_JFK,0.146075,0.323912,0.219178
1,ATL,AA,ATL_CLT,0.146075,0.178146,0.174757
2,ATL,AA,ATL_DFW,0.146075,0.178146,0.191919
3,ATL,AA,ATL_ORD,0.146075,0.178146,0.153153
4,ATL,DL,ATL_FLL,0.146075,0.143742,0.168750


## Random Forest Comparison

This compares the flight-only version against the flight-plus-weather version.


In [27]:
flight_numeric_features = [
    "DayOfWeek",
    "dep_hour",
    "is_weekend",
    "origin_delay_rate",
    "airline_delay_rate",
    "route_delay_rate"
]

flight_categorical_features = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "route",
    "time_of_day_bin"
]

weather_numeric_features = [
    "tmpf",
    "dwpf",
    "relh",
    "sknt",
    "gust",
    "alti",
    "p01i",
    "vsby",
    "weather_report_age_minutes",
    "has_precip",
    "low_visibility",
    "high_wind",
    "has_gust",
    "has_weather_code"
]

weather_categorical_features = ["skyc1"]

flight_only_features = flight_numeric_features + flight_categorical_features
flight_weather_features = flight_only_features + weather_numeric_features + weather_categorical_features

target = "Delay"

In [28]:
def build_random_forest_pipeline(numeric_features, categorical_features):
    preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="most_frequent"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=12,
            min_samples_leaf=5,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    return model


experiments = {
    "Flight Only": {
        "features": flight_only_features,
        "numeric": flight_numeric_features,
        "categorical": flight_categorical_features,
    },
    "Flight Plus Weather": {
        "features": flight_weather_features,
        "numeric": flight_numeric_features + weather_numeric_features,
        "categorical": flight_categorical_features + weather_categorical_features,
    },
}

results = []

for experiment_name, spec in experiments.items():
    X_train = train_df[spec["features"]]
    y_train = train_df[target]
    X_test = test_df[spec["features"]]
    y_test = test_df[target]

    model = build_random_forest_pipeline(spec["numeric"], spec["categorical"])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "experiment": experiment_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_delay": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_delay": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_delay": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    })

    print("=" * 80)
    print(experiment_name)
    print(classification_report(y_test, y_pred, zero_division=0))
    print(confusion_matrix(y_test, y_pred))

pd.DataFrame(results).sort_values("f1_delay", ascending=False)

Flight Only
              precision    recall  f1-score   support

           0       0.85      0.65      0.73     29617
           1       0.30      0.56      0.39      7898

    accuracy                           0.63     37515
   macro avg       0.57      0.60      0.56     37515
weighted avg       0.73      0.63      0.66     37515

[[19163 10454]
 [ 3510  4388]]
Flight Plus Weather
              precision    recall  f1-score   support

           0       0.85      0.69      0.76     29617
           1       0.32      0.54      0.40      7898

    accuracy                           0.66     37515
   macro avg       0.58      0.62      0.58     37515
weighted avg       0.74      0.66      0.68     37515

[[20379  9238]
 [ 3615  4283]]


,experiment,accuracy,precision_delay,recall_delay,f1_delay
1,Flight Plus Weather,0.657390,0.316767,0.542289,0.399925
0,Flight Only,0.627776,0.295647,0.555584,0.385928
